In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip "/content/drive/MyDrive/VisDrone/VisDrone2019-DET-train.zip" -d /content/
!unzip "/content/drive/MyDrive/VisDrone/VisDrone2019-DET-val.zip" -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000240.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000241.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000242.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000243.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000244.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000245.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000246.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000247.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000248.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000249.jpg  
  inflating: /content/VisDrone2019-DET-train/images/9999951_00000_d_0000250.jpg  
  inflating: /content/VisDrone2019-DET-train/im

In [ ]:
!pip install ultralytics opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.4 MB/s eta 0:00:00


In [ ]:
import os
import shutil
from PIL import Image

def convert_visdrone_to_yolo(images_dir, annotations_dir, output_dir, max_images=None):
    os.makedirs(output_dir + "/images", exist_ok=True)
    os.makedirs(output_dir + "/labels", exist_ok=True)

    class_map = {
        1: 0, 2: 1, 3: 2, 4: 3, 5: 4,
        6: 5, 7: 6, 8: 7, 9: 8, 10: 9
    }

    files = os.listdir(images_dir)
    if max_images:
        files = files[:max_images]

    for file in files:
        if not file.endswith(".jpg"):
            continue

        img_path = os.path.join(images_dir, file)
        ann_path = os.path.join(annotations_dir, file.replace(".jpg", ".txt"))

        shutil.copy(img_path, output_dir + "/images/" + file)

        if not os.path.exists(ann_path):
            continue

        img = Image.open(img_path)
        img_w, img_h = img.size

        yolo_labels = []

        with open(ann_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split(',')
            if len(parts) < 6:
                continue
            x, y, w, h, score, cls = map(int, parts[:6])
            if cls not in class_map:
                continue
            cls_id = class_map[cls]
            xc = (x + w / 2) / img_w
            yc = (y + h / 2) / img_h
            w = w / img_w
            h = h / img_h
            yolo_labels.append(f"{cls_id} {xc} {yc} {w} {h}")

        label_path = os.path.join(output_dir, "labels", file.replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            f.write("\n".join(yolo_labels))

train_images = "/content/VisDrone2019-DET-train/images"
train_anns = "/content/VisDrone2019-DET-train/annotations"
val_images = "/content/VisDrone2019-DET-val/images"
val_anns = "/content/VisDrone2019-DET-val/annotations"

convert_visdrone_to_yolo(train_images, train_anns, "/content/dataset/train")

convert_visdrone_to_yolo(val_images, val_anns, "/content/dataset/val")

In [ ]:
data_yaml = """
train: /content/dataset/train/images
val: /content/dataset/val/images

nc: 10
names: ['pedestrian','people','bicycle','car','van','truck','tricycle','awning-tricycle','bus','motor']
"""

with open("/content/dataset/data.yaml", "w") as f:
    f.write(data_yaml)

In [ ]:
import os

train_images_dir = "/content/dataset/train/images"
train_labels_dir = "/content/dataset/train/labels"

print("عدد صور التدريب:", len(os.listdir(train_images_dir)))
print("عدد ملفات labels:", len(os.listdir(train_labels_dir)))

val_images_dir = "/content/dataset/val/images"
val_labels_dir = "/content/dataset/val/labels"

print("عدد صور validation:", len(os.listdir(val_images_dir)))
print("عدد ملفات labels validation:", len(os.listdir(val_labels_dir)))

عدد صور التدريب: 6471
عدد ملفات labels: 6471
عدد صور validation: 548
عدد ملفات labels validation: 548


In [ ]:
from ultralytics import YOLO

data_path = "/content/dataset/data.yaml"
model = YOLO("yolo11m.pt")

model.train(
    data=data_path,
    epochs=100,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=0.001,

    mosaic=1.0,
    mixup=0.15,

    cache=False,
    amp=True,
    project='/content/drive/MyDrive/VisDrone/results',
    name='yolo11_m'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=

In [ ]:
from ultralytics import YOLO

last_weights = '/content/drive/MyDrive/VisDrone/results/yolo11_m/weights/last.pt'


model = YOLO(last_weights)

model.train(resume=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgs

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2c96d34b60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
model_path = '/content/drive/MyDrive/VisDrone/results/yolo11_m/weights/best.pt'
model = YOLO(model_path)

metrics = model.val(
    data="/content/dataset/data.yaml",
    imgsz=640,
    batch=8,
    conf=0.001,
    iou=0.6,
    device=0,
    save_json=True
)

print("-" * 30)
print(f"النتائج النهائية لمشروع EagleEye:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print("-" * 30)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,037,742 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2118.2±895.9 MB/s, size: 126.7 KB)
val: Scanning /content/dataset/val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 191.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 4.0it/s 17.2s
                   all        548      38759      0.573      0.469      0.463      0.277
            pedestrian        520       8844      0.631      0.501      0.514      0.235
                people        482       5125      0.619       0.37      0.379      0.145
               bicycle        364       1287      0.341      0.258      0.215      0.098
                   car        515      14064      0.779      0.827      0.828      0.584
                   van        421       197

In [ ]:
from ultralytics import YOLO

model_path = '/content/drive/MyDrive/VisDrone/results/yolo11_m/weights/best.pt'
model = YOLO(model_path)

model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=0.0005,
    patience=10,

    mosaic=1.0,
    mixup=0.15,

    amp=True,
    project='/content/drive/MyDrive/VisDrone/results',
    name='yolo11_m_refined'
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/content/drive/MyDrive/VisDrone/results/yolo11_m/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11_m_refined, nbs=64, nms=False, opset=None, opt

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2d22266d80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
model = YOLO("/content/drive/MyDrive/VisDrone/yolo11-p2.yaml").load("/content/drive/MyDrive/VisDrone/results/yolo11_m/weights/best.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=6,
    optimizer="AdamW",
    lr0=0.001,
    patience=10,
    mosaic=1.0,
    mixup=0.15,
    amp=True,
    project='/content/drive/MyDrive/VisDrone/results',
    name='EagleEye_P2_Final_Run'
)

WARNING ⚠️ no model scale passed. Assuming scale='m'.
Transferred 427/803 items from pretrained weights
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/content/drive/MyDrive/VisDrone/yolo11-p2.yaml, momentum=0.937, m

In [ ]:
from ultralytics import YOLO

last_model_path = '/content/drive/MyDrive/VisDrone/results/EagleEye_P2_Final_Run/weights/last.pt'

model = YOLO(last_model_path)

model.train(resume=True)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/content/drive/MyDrive/VisDrone/results/EagleEye_P2_Final_Run/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=EagleEye_P2_Final_Run, nbs=64, nms=False

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x784479a1dfa0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
from ultralytics import YOLO

model_path = '/content/drive/MyDrive/VisDrone/results/EagleEye_P2_Final_Run/weights/best.pt'

model = YOLO(model_path)

model.train(
    data="/content/dataset/data.yaml",
    epochs=15,
    imgsz=640,
    batch=6,
    warmup_epochs=0,
    lr0=0.0001,
    optimizer="AdamW",
    project='/content/drive/MyDrive/VisDrone/results',
    name='EagleEye_P2_Rescue_Run',
    close_mosaic=10
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/VisDrone/results/EagleEye_P2_Final_Run/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=EagleEye_P2_Rescue_Run, nbs=64, nms=Fals

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7845de93f260>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0